## For GC controls with coordinates from hg38 and seperate region.bed of 80K NGN2 derived neurons
- `GC_Hon`, `GC_Vista`, `GC_Cort_Chengyu`, `GC_GABA_Chengyu`, `GC_Glut_Chengyu`
- 6, 253, 185, 40, 85
- within `GC_Cort_Chengyu` there are still ',' at positions where in the new file are '~' because of problems with ',' in the header
- these are in the design region file as well

In [1]:
from importlib import reload
import pandas as pd
import sys
import os
sys.path.append('../helpful_functions')
import helpful_functions as hf
reload(hf)

sample_tsv_path = '/home/kisa/coding/80K_MPRA/80K-Analysis/05_variant_region_list/resources/regions/controls/design.control_regions.tsv' # tsv with sample\tbed_path

In [2]:
# column names
col_name = 'name'
col_header = 'name'
col_sequence = 'sequence'
col_category = 'category'
col_class = 'class'
col_source = 'source'
col_ref = 'ref'
col_chr = 'chr'
col_start = 'start'
col_end = 'end'
col_strand = 'strand'
col_variant_class = 'variant_class'
col_variant_pos = 'variant_pos'
col_SPDI = 'SPDI'
col_allele = 'allele'
col_info = 'info'
my_col_ref_base = 'tmp_ref_base'
my_col_alt_base = 'tmp_alt_base'


interesting_columns = [col_name, col_sequence, col_category, col_class, col_source, col_ref,
                       col_chr, col_start, col_end, col_strand, col_variant_class, col_variant_pos, col_SPDI, col_allele, col_info]

In [3]:
# read dict for paths:
with open(sample_tsv_path, 'r') as file:
    lines = file.readlines()
    group_bed_path_dict = {}
    for line in lines:
        key, value = line.strip().split('\t')
        group_bed_path_dict[key.strip()] = value.strip()

### GC_Hon
- how many? 6 
- are any variants in these sequences? No => No spdi 


In [10]:
import yaml

# config
config_path = "/home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/notebooks/control_metadata/config_file.yaml"
with open(config_path) as conf:
    config = yaml.load(conf, Loader=yaml.FullLoader)
    conf.close()

input_fasta = config['design_file']
pre_metadata_df = hf.fasta_to_dataframe(input_fasta, columns=[col_name, col_sequence])
# split the metadata file headers by '#'
# Apply the function to each row and concatenate the results
pre_metadata_df_split = pd.concat(pre_metadata_df.apply(lambda row: hf.split_ids(row, id_col=col_name, separator='#'), axis=1).values)

# Reset the index
pre_metadata_df_split.reset_index(drop=True, inplace=True)

pre_metadata_df = pre_metadata_df_split.copy()
print(pre_metadata_df.shape[0]) # 80804

pre_metadata_df['tmp_label'] = pre_metadata_df[col_name].apply(lambda x: hf.get_label(x))

# # filter for group to get an overview
pre_metadata_df_group = pre_metadata_df.loc[pre_metadata_df['tmp_label'] == 'GC_Hon'].copy()
pre_metadata_df_group

80804


,name,sequence,tmp_label
75284,GC_Hon:TBXEnh4_Centered_region|chr12:114025966...,AGGACCGGATCAACTGGGCTCACTGTCCTTGGTGAATGACTTACAT...,GC_Hon
75285,GC_Hon:TBXEnh2_EH38E1646265|chr12:114335696-11...,AGGACCGGATCAACTTGACACTATCGGCCTGTTGTCTAAAGTACTT...,GC_Hon
75286,GC_Hon:TBXEnh3_EH38E3042928|chr12:114361591-11...,AGGACCGGATCAACTTTCCAAACACAAGGAGCCCTCCGGCCTGCCT...,GC_Hon
75287,GC_Hon:TBXEnh6_EH38E1646350|chr12:114415524-11...,AGGACCGGATCAACTCTTTTATGATTATTATTTTTACCGACAGCTC...,GC_Hon
75288,GC_Hon:TBXEnh6_EH38E3042962|chr12:114415802-11...,AGGACCGGATCAACTGAAATAGCAGGTGCAAGATCCATGCTGGGCA...,GC_Hon
75289,GC_Hon:NGFRAP1_Centered_region|chrX:103376085-...,AGGACCGGATCAACTAGAAGGGTAGGAATTCAGGGGCTACGTGGGC...,GC_Hon


In [11]:
# add the columns of the metadata file
pre_metadata_df[col_category] = 'NA'
pre_metadata_df[col_class] = ''
pre_metadata_df[col_source] = 'NA'
pre_metadata_df[col_ref] = 'GRCh38' # GRCh37
pre_metadata_df[col_chr] = 'NA'
pre_metadata_df[col_start] = 'NA'
pre_metadata_df[col_end] = 'NA'
pre_metadata_df[col_strand] = 'NA'
pre_metadata_df[col_variant_class] = 'NA'
pre_metadata_df[col_variant_pos] = 'NA'
pre_metadata_df[col_SPDI] = 'NA'
pre_metadata_df[col_allele] = 'NA'
pre_metadata_df[col_info] = '' # 'Coordinates are based on GRCh37 (wrong genome build)'

# remove adapter from sequence (15bp of start and end):
pre_metadata_df[col_sequence] = pre_metadata_df[col_sequence].apply(lambda x: x[15:-15])


In [12]:
def split_ids(row, id_col, separator=';'):
    """Function to split the name and create new rows while conserving all columns"""
    ids = row[id_col].split(separator)
    new_rows = []
    for id in ids:
        new_row = row.copy()
        new_row[id_col] = id
        new_rows.append(new_row)
    return pd.DataFrame(new_rows)

def get_bed_coordinates(pre_metadata_df, group_name, bed_file_path, output_path, interesting_columns=[col_name, col_sequence, col_category, col_class, col_source, col_ref, col_chr, col_start, col_end, col_strand, col_variant_class, col_variant_pos, col_SPDI, col_allele, col_info], with_comma=False):
    """Function getting a metadata df, filters this for the group_name and reads and joins it with the coordinates from the given bed file before it is writing it to output_path"""
    # add name_no_label
    pre_metadata_df['name_no_label'] = pre_metadata_df['name'].apply(lambda name: ':'.join(name.split(':')[1:]))
    # filter metadata_df to only have the bed file with the desired label:
    pre_metadata_df_filtered = pre_metadata_df.loc[pre_metadata_df['tmp_label'] == group_name].copy()

    # error if # in name (because this means matched header because of same sequence)
    if pre_metadata_df_filtered[col_name].str.contains('#').sum() != 0:
        print("Names with seperator char '#' detected" )
        try:
            # Apply the function to each row and concatenate the results
            pre_metadata_df_group_split = pd.concat(pre_metadata_df_filtered.apply(lambda row: split_ids(row, id_col=col_name, separator='#'), axis=1).values)

            # Reset the index
            pre_metadata_df_filtered = pre_metadata_df_group_split.reset_index(drop=True, inplace=False)
            pre_metadata_df_filtered['name_no_label'] = pre_metadata_df_filtered['name'].apply(lambda name: ':'.join(name.split(':')[1:]))
        except:
            raise ValueError("Row names contain separator char '#', split rows first and continue")


    bed_file = pd.read_csv(bed_file_path, sep="\t", comment="#", header=None)
    prefix = 'bed'
    bed_file.columns = [ f'{prefix}_{column}' for column in [col_chr, col_start, col_end, col_name, 'score', col_strand]]
    matching_column = f'{prefix}_name'
    if with_comma:
        # replace ',' with '~' for all
        bed_file[f'{prefix}_name_mod'] = bed_file[matching_column].str.replace(',', '~')
        matching_column = f'{prefix}_name_mod'
    merged_coordinates = pre_metadata_df_filtered.merge(bed_file, left_on='name_no_label', right_on=matching_column, how='inner')
    merged_coordinates[col_category] = 'element'
    merged_coordinates[col_class] = 'element inactive control'
    # drop chr, start, end
    merged_coordinates.drop(columns=[col_chr, col_start, col_end], inplace=True)

    merged_coordinates.rename(columns={f'{prefix}_{col_chr}': col_chr, f'{prefix}_{col_start}': col_start, f'{prefix}_{col_end}': col_end}, inplace=True)
    merged_coordinates = merged_coordinates[interesting_columns]

    # check if the row sum is equal to the initial row number
    print(f'Expected number of rows: {pre_metadata_df_filtered[col_sequence].nunique()}')
    print(f'Number of rows with matching coordinates: {merged_coordinates.shape[0]}')

    # Write DataFrame to TSV file
    merged_coordinates[interesting_columns].to_csv(os.path.join(output_path, f'{group_name}.metadata.tmp.tsv.gz'), sep='\t', index=False, na_rep='NA', compression='gzip')
    os.system(f'zcat {output_path}/{group_name}.metadata.tmp.tsv.gz | sed "s/\'/\\"/g" | gzip -c > {output_path}/{group_name}.metadata.tsv.gz')
    return pre_metadata_df_filtered


In [17]:
group_name = 'GC_Hon'
output_dir = config['final_output_dir']
bed_file_path = group_bed_path_dict[group_name]
output_path = f'/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/{group_name}'
output_path = '/'.join(bed_file_path.split('/')[:-1])
output_path = os.path.join(output_dir, group_name)

get_bed_coordinates(pre_metadata_df, group_name, bed_file_path, output_path, interesting_columns)
# # filter metadata_df to only have the bed file with the desired label:
# pre_metadata_df_filtered = pre_metadata_df.loc[pre_metadata_df['tmp_label'] == group_name].copy()

# bed_file = pd.read_csv(bed_file_path, sep="\t", comment="#", header=None)
# prefix = 'bed'
# bed_file.columns = [ f'{prefix}_{column}' for column in [col_chr, col_start, col_end, col_name, 'score', col_strand]]
# merged_coordinates = pre_metadata_df_filtered.merge(bed_file, left_on="name_no_label_rstrip", right_on=f'{prefix}_name', how='inner')
# merged_coordinates[col_category] = 'element'
# # drop chr, start, end
# merged_coordinates.drop(columns=[col_chr, col_start, col_end], inplace=True)

# merged_coordinates.rename(columns={f'{prefix}_{col_chr}': col_chr, f'{prefix}_{col_start}': col_start, f'{prefix}_{col_end}': col_end}, inplace=True)
# merged_coordinates = merged_coordinates[interesting_columns]

# # check if the row sum is equal to the initial row number
# print(f'Expected number of rows: {pre_metadata_df_filtered.shape[0]}')
# print(f'Number of rows with matching coordinates: {merged_coordinates.shape[0]}')

# # Write DataFrame to TSV file
# merged_coordinates[interesting_columns].to_csv(os.path.join(output_path, f'{group_name}.metadata.tmp.tsv.gz'), sep='\t', index=False, na_rep='NA', compression='gzip')
# os.system(f'zcat {output_path}/{group_name}.metadata.tmp.tsv.gz | sed "s/\'/\\"/g" | gzip -c > {output_path}/{group_name}.metadata.tsv.gz')

Expected number of rows: 6
Number of rows with matching coordinates: 6


,name,sequence,tmp_label,category,class,source,ref,chr,start,end,strand,variant_class,variant_pos,SPDI,allele,info,name_no_label
75284,GC_Hon:TBXEnh4_Centered_region|chr12:114025966...,GGGCTCACTGTCCTTGGTGAATGACTTACATCTGAGCTTCTTCCTG...,GC_Hon,NA,,NA,GRCh38,NA,NA,NA,NA,NA,NA,NA,NA,,TBXEnh4_Centered_region|chr12:114025966-114026215
75285,GC_Hon:TBXEnh2_EH38E1646265|chr12:114335696-11...,TGACACTATCGGCCTGTTGTCTAAAGTACTTACAGTTCTCTCCGGG...,GC_Hon,NA,,NA,GRCh38,NA,NA,NA,NA,NA,NA,NA,NA,,TBXEnh2_EH38E1646265|chr12:114335696-114335945
75286,GC_Hon:TBXEnh3_EH38E3042928|chr12:114361591-11...,TTCCAAACACAAGGAGCCCTCCGGCCTGCCTGGCCCTGTGACATTT...,GC_Hon,NA,,NA,GRCh38,NA,NA,NA,NA,NA,NA,NA,NA,,TBXEnh3_EH38E3042928|chr12:114361591-114361840
75287,GC_Hon:TBXEnh6_EH38E1646350|chr12:114415524-11...,CTTTTATGATTATTATTTTTACCGACAGCTCCAAGCAGAGGCCTGT...,GC_Hon,NA,,NA,GRCh38,NA,NA,NA,NA,NA,NA,NA,NA,,TBXEnh6_EH38E1646350|chr12:114415524-114415773
75288,GC_Hon:TBXEnh6_EH38E3042962|chr12:114415802-11...,GAAATAGCAGGTGCAAGATCCATGCTGGGCACGAACACGGGCAGAA...,GC_Hon,NA,,NA,GRCh38,NA,NA,NA,NA,NA,NA,NA,NA,,TBXEnh6_EH38E3042962|chr12:114415802-114416051
75289,GC_Hon:NGFRAP1_Centered_region|chrX:103376085-...,AGAAGGGTAGGAATTCAGGGGCTACGTGGGCATTGATGGCTGTGCA...,GC_Hon,NA,,NA,GRCh38,NA,NA,NA,NA,NA,NA,NA,NA,,NGFRAP1_Centered_region|chrX:103376085-103376334


### Identify all controls without variants: according to [this document](https://docs.google.com/spreadsheets/d/1ilr3Fc97SKBTYWsCuS1OJqN--E46QPoev28Z7CBZq4k/edit?usp=sharing): these GC_controls are only element controls

In [22]:
group_names = ['GC_Vista', 'GC_Cort_Chengyu', 'GC_GABA_Chengyu', 'GC_Glut_Chengyu']
output_dir = config['final_output_dir']

for group_name in group_names:
    bed_file_path = group_bed_path_dict[group_name]
    output_path = f'/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/mpra_metadata_format/{group_name}'
    output_path = '/'.join(bed_file_path.split('/')[:-1])
    output_path = os.path.join(output_dir, group_name)

    if group_name == 'GC_Cort_Chengyu':
        get_bed_coordinates(pre_metadata_df, group_name, bed_file_path, output_path, interesting_columns, with_comma=False)
    elif group_name == 'GC_GABA_Chengyu':
        GC_GABA_Chengyu = get_bed_coordinates(pre_metadata_df, group_name, bed_file_path, output_path, interesting_columns, with_comma=False)
    else:
        get_bed_coordinates(pre_metadata_df, group_name, bed_file_path, output_path, interesting_columns, with_comma=False)

Expected number of rows: 256
Number of rows with matching coordinates: 256
Expected number of rows: 185
Number of rows with matching coordinates: 185
Expected number of rows: 85
Number of rows with matching coordinates: 85
Expected number of rows: 83
Number of rows with matching coordinates: 83
